# 02 — Topic Modeling: TF-IDF + K-Means

Loads the preprocessed `.npz` from the data pipeline, applies TF-IDF vectorization, uses the elbow method to select K, fits K-Means clustering, inspects top terms per cluster, and visualizes results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

%matplotlib inline
sns.set_theme(style='whitegrid')

PROJECT_ROOT  = Path.cwd().parent
NPZ_PATH      = PROJECT_ROOT / 'data' / 'processed' / 'reviews.npz'

print('NPZ path:', NPZ_PATH)

## 1. Load `.npz`

In [ ]:
assert NPZ_PATH.exists(), f'NPZ file not found at {NPZ_PATH}. Run 01_data_pipeline.ipynb first.'

data = np.load(NPZ_PATH, allow_pickle=True)
print('Arrays in NPZ:', list(data.keys()))

reviews = data['reviews'].astype(str)
ids     = data['ids']     if 'ids'     in data else np.arange(len(reviews))
ratings = data['ratings'] if 'ratings' in data else None

print(f'Loaded {len(reviews):,} reviews')
print('Sample:', reviews[:3])

## 2. TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
)

X = vectorizer.fit_transform(reviews)
print(f'TF-IDF matrix shape: {X.shape}  (reviews × features)')

## 3. Choose K — Elbow Method

Plot inertia (within-cluster sum of squares) for K = 2 to 15. Look for the elbow point.

In [ ]:
K_RANGE = range(2, 16)
inertias = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=5, random_state=42, max_iter=200)
    km.fit(X)
    inertias.append(km.inertia_)
    print(f'  K={k:2d}  inertia={km.inertia_:.0f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(K_RANGE), inertias, marker='o', linewidth=2)
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method — Optimal K Selection')
ax.set_xticks(list(K_RANGE))
plt.tight_layout()
plt.show()

## 4. Fit K-Means with Chosen K

Set `K` below based on the elbow plot above.

In [ ]:
# ---- SET THIS after inspecting the elbow plot ----
K = 6
# --------------------------------------------------

kmeans = KMeans(n_clusters=K, init='k-means++', n_init=10, random_state=42, max_iter=300)
kmeans.fit(X)

labels = kmeans.labels_
print(f'Fitted K-Means with K={K}')
print('Cluster sizes:', pd.Series(labels).value_counts().sort_index().to_dict())

## 5. Top TF-IDF Terms per Cluster

In [ ]:
N_TOP = 15
feature_names = vectorizer.get_feature_names_out()
centroids = kmeans.cluster_centers_

for cluster_id in range(K):
    top_indices = centroids[cluster_id].argsort()[-N_TOP:][::-1]
    top_terms   = [feature_names[i] for i in top_indices]
    count       = (labels == cluster_id).sum()
    print(f'Cluster {cluster_id} ({count:,} reviews):')
    print('  ', ' | '.join(top_terms))
    print()

## 6. Assign Cluster Labels to Reviews

In [ ]:
df_results = pd.DataFrame({
    'id':      ids,
    'review':  reviews,
    'cluster': labels,
})

if ratings is not None:
    df_results['rating'] = ratings

print(df_results.head())
print(f'\nCluster distribution:')
print(df_results['cluster'].value_counts().sort_index())

## 7. Visualize Clusters

### 7a. PCA Scatter Plot (2D projection)

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X.toarray())

fig, ax = plt.subplots(figsize=(10, 7))
palette = sns.color_palette('tab10', K)

for cluster_id in range(K):
    mask = labels == cluster_id
    ax.scatter(
        coords[mask, 0], coords[mask, 1],
        c=[palette[cluster_id]], label=f'Cluster {cluster_id}',
        alpha=0.4, s=10, linewidths=0,
    )

ax.set_title(f'K-Means Clusters (K={K}) — PCA 2D Projection')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

### 7b. Cluster Size Bar Chart

In [ ]:
cluster_counts = df_results['cluster'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    [f'Cluster {i}' for i in cluster_counts.index],
    cluster_counts.values,
    color=palette,
)
ax.bar_label(bars, fmt='%d')
ax.set_title('Number of Reviews per Cluster')
ax.set_xlabel('Cluster')
ax.set_ylabel('Review Count')
plt.tight_layout()
plt.show()

## 8. Sample Reviews per Cluster

Qualitative check — do reviews within each cluster share a coherent theme?

In [ ]:
N_SAMPLES = 5

for cluster_id in range(K):
    subset = df_results[df_results['cluster'] == cluster_id]
    sample = subset.sample(min(N_SAMPLES, len(subset)), random_state=42)
    print(f'=== Cluster {cluster_id} ({len(subset):,} reviews) ===')
    for _, row in sample.iterrows():
        # Show first 200 characters of the processed review
        print(f'  • {row["review"][:200]}')
    print()